# Stage 1: inventory

Turns the downloads into a fixed, fully described list of photos that every later stage builds on, and
settles whether photos are tiled at native resolution or rescaled first (Stage 1).

All logic lives in `src/inventory.py`, `src/leaf_width.py` and `src/inventory_report.py`; this notebook
only calls them. Outputs: `data/inventory.csv`, `data/paddy_inventory.csv`, `data/leaf_width_selection.csv`
and `docs/reports/inventory_report.md`.

In [ ]:
from IPython.display import Markdown, display

from src.config import load_config
from src.inventory import write_inventories
from src.inventory_report import write_report
from src.leaf_width import evaluate, freeze_selection

config = load_config("configs/inventory.yaml")

## 1. Inventory of both datasets

Every file is listed (not only images), with checksum, perceptual hash, size, camera, capture time and JPEG quality. About a minute on 16 CPU threads.

In [ ]:
inventories = write_inventories(config)

## 2. Photos for the pixel-scale check

The 30 photos are chosen once (one per equal-width band of photo size) and frozen, so every measurer sees the same set.

In [ ]:
selected = freeze_selection(config)  # reads the frozen file if it already exists
selected

## 3. Report

In [ ]:
display(Markdown(write_report(config)))

## 4. Leaf-width measurement (two raters, in a terminal)

Two measurers each run, on their own and without looking at the other's numbers (2026-11-07: Ahsan and
`model_rater_a`, a model rater that measured through the same tool code; the report names it as such):

```
uv run python -m src.leaf_width measure --measurer <name>
```

Then `uv run python -m src.leaf_width decide` applies the pre-declared rule and writes
`data/pixel_scale_decision.json`; re-run this notebook and section 8 of the report fills in.

In [ ]:
result = evaluate(config)  # reads only; `python -m src.leaf_width decide` writes the decision
print(result.get("reason") or (f"rho = {result['spearman_rho']:.3f} -> rescale: {result['rescale']}"
                               if result["complete"] else f"pending: {result['answered'] or 'nobody yet'}"))